# Reproduce Identity2Vec (I2V)

**How to use this notebook:** click each cell in order from top to bottom, and press **Shift+Enter** to run it. Read the short note above each cell. That's it — you don't type any terminal commands.

Goal: take the I2V embeddings and reproduce the paper's **scores** on Cora.

Each code cell just calls a function from the project files — you are not writing code.

## Step 0 — Set up
Tells Python where the project is. Just run it once.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)                       # run everything from the project root
for p in (str(ROOT), str(ROOT / 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Working from:', Path.cwd())

## Step 1 — Look at the graph
A graph is dots (**nodes**) joined by lines (**edges**). Let's load Cora and count them.

In [ ]:
import networkx as nx
G = nx.read_edgelist('input/cora.edgelist', nodetype=int)
print('Cora:  nodes =', G.number_of_nodes(), ' edges =', G.number_of_edges())

## Step 2 — Load the embeddings
I2V turns each node into a list of 64 numbers (its 'fingerprint'). Making them is slow, so we use the author's ready-made file `output/cora.emb`.

(Want to make your own? There's an optional slow cell at the very bottom.)

In [ ]:
from utils import load_embeddings
emb = load_embeddings('output/cora.emb')
print('Loaded', len(emb.index_to_key), 'node fingerprints, each of length', emb.vector_size)
print('Example - node 164, first 5 numbers:', emb['164'][:5])

## Step 3 — Get the answer-keys (labels)
For the 'guess each node's category' test we need each node's **true category**. This cell downloads the official Cora and **safely checks** the node IDs match our graph before writing `labels/cora.labels`.

**Needs internet.** If it times out, just run the cell again. (If it keeps failing, tell me and we'll fetch the labels another way.)

In [ ]:
from make_planetoid_labels import make_labels
make_labels('Cora')     # prints edge_overlap and 'WROTE labels/cora.labels' on success

## Step 4 — Reproduce the node-classification score
The real test: using the fingerprints, can a simple model guess each node's category? The score is **weighted F1** (0 to 1, higher is better).

Compare it to the paper's Cora number. Within about **0.03** counts as reproduced.

In [ ]:
from eval_nodeclass import evaluate
f1, n, n_classes = evaluate('output/cora.emb', 'labels/cora.labels', train_frac=0.8, seed=42)
print(f'Cora node classification:  weighted F1 = {f1:.4f}   ({n} nodes, {n_classes} categories)')

## Step 5 — Save the result
Writes a small score sheet into `results/` so it's recorded with its settings.

In [ ]:
from results_io import save_result
save_result('results', 'cora', 'nodeclass',
            {'weighted_f1': f1, 'n_nodes': n, 'n_classes': n_classes},
            {'seed': 42, 'train_frac': 0.8, 'emb': 'cora.emb'})

## Step 6 (optional) — Link prediction
A second test: hide 30% of the edges and ask the model to guess them back (score = **AUC**).

Done correctly this needs a **fresh embedding trained on only the remaining 70%** (otherwise it cheats). That training is slow right now, so this cell just makes the split; the slow training step is noted below it.

In [ ]:
from prepare_linkpred import prepare
counts = prepare('input/cora.edgelist', 'cora', 'splits', test_frac=0.3, seed=42)
print(counts)
print('\nNext (slow, optional): train an embedding on splits/cora_train.edgelist,')
print('then: from eval_linkpred import evaluate; evaluate(<that emb>, "splits", "cora")')

## (Optional) Make your own embedding - SLOW
This runs the I2V trainer yourself. It can take a long time (the speed fix is a separate task). Uncomment the last line to run.

In [ ]:
from runner import embed
# embed('input/cora.edgelist', 'output/cora_mine.emb')   # <- remove the # to run (slow)